# Train pair-judge cross-encoder on Colab — v2

Trains a binary `(tz_req, pmi_unit) → P(match)` classifier on `c_pairs_variant_a.csv`.

**v1 → v2 changes** (v1 gave test F1 = 0.59, recall on MATCH only 58%):
- base → `xlm-roberta-base` (stronger Russian than mBERT)
- `--auto-pos-weight` → up-weights MATCH class ~11× in CE loss to fix 8% class imbalance
- epochs 4, lr 1e-5

**Before running:** in `My Drive/c-quality/` have `train_pair_judge.py` and `c_pairs_variant_a.csv`. Base model now downloads from HF, no upload needed.

**Runtime → Change runtime type → T4 GPU**.

## 1. Mount Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi -L
!ls /content/drive/MyDrive/c-quality/

## 2. Copy script + data to local disk

In [ ]:
!mkdir -p /content/work
!cp /content/drive/MyDrive/c-quality/train_pair_judge.py /content/work/
!cp /content/drive/MyDrive/c-quality/c_pairs_variant_a.csv /content/work/
!ls /content/work/

## 3. Verify environment

In [ ]:
import torch, transformers, sklearn, pandas
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('transformers:', transformers.__version__)
print('sklearn:', sklearn.__version__)
print('pandas:', pandas.__version__)

## 4. Baseline eval of XLM-R base on pair task (~1 min on T4)

Expected F1 very low — XLM-R is not trained for pair classification out of the box.

In [ ]:
%cd /content/work
!python train_pair_judge.py --eval-only \
    --base xlm-roberta-base \
    --csv  ./c_pairs_variant_a.csv \
    --out  ./pair_judge_v2 \
    --batch 32 --max-len 256

## 5. Fine-tune v2 (~20-30 min on T4)

**Targets:**
- `[final] test F1` ≥ 0.75 (realistic), stretch 0.85
- `pseudo_pos` accuracy up from 58% → 75%+
- threshold sweep should spread (v1 was flat at 0.59 across all thresholds)

In [ ]:
%cd /content/work
!python train_pair_judge.py \
    --base xlm-roberta-base \
    --csv  ./c_pairs_variant_a.csv \
    --out  ./pair_judge_v2 \
    --batch 16 --grad-accum 2 --epochs 4 --lr 1e-5 --max-len 256 \
    --auto-pos-weight

## 6. Sanity check — the same 5 pairs as v1

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

path = '/content/work/pair_judge_v2'
tok = AutoTokenizer.from_pretrained(path)
m = AutoModelForSequenceClassification.from_pretrained(path).cuda().eval()

pairs = [
    # Clear MATCH — expect p > 0.8
    ('Система должна хранить журнал событий не менее 90 дней.',
     'Проверить, что журнал операций сохраняется в течение 90 дней.'),
    # Clear NO-MATCH — expect p < 0.1
    ('Система должна хранить журнал событий не менее 90 дней.',
     'Пользователь может изменить цветовую схему интерфейса.'),
    # Partial / same topic
    ('Время отклика интерфейса не должно превышать 2 секунд.',
     'Проверить время отклика системы при нагрузке.'),
    # CONFLICT case — rule-based, model may still say match
    ('Система должна обеспечивать доступность не менее 99.9%.',
     'Система должна обеспечивать доступность не менее 95%.'),
    # Paraphrase — expect p > 0.6
    ('Доступ администратора защищён двухфакторной аутентификацией.',
     'Администратор входит в систему через 2FA.'),
]
with torch.no_grad():
    for req, unit in pairs:
        enc = tok(req, unit, return_tensors='pt', truncation=True, max_length=256).to('cuda')
        p = torch.softmax(m(**enc).logits[0], dim=-1).cpu().tolist()
        verdict = 'MATCH  ' if p[1] > 0.5 else 'no     '
        print(f'  p_match={p[1]:.3f}  [{verdict}]')
        print(f'    REQ:  {req}')
        print(f'    UNIT: {unit}')
        print()

## 7. Save checkpoint back to Drive

In [ ]:
!rm -rf /content/drive/MyDrive/c-quality/pair_judge_v2
!cp -r /content/work/pair_judge_v2 /content/drive/MyDrive/c-quality/
!ls -la /content/drive/MyDrive/c-quality/pair_judge_v2/